<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Freshness Multiplier

The paper reports that mature content refreshed within the last 30 days had much stronger observed performance than older content that had not been refreshed recently. In the portfolio, refreshed 365+ day content showed about a 3.2x higher health score and 57x more impressions.

My methodology question is about how pages were selected for refresh. Were the refreshed pages already stronger, more visible, or more strategically important before the update? If so, part of the difference may come from selection bias rather than the refresh itself.

I would want to compare refreshed pages with similar non-refreshed pages matched on prior impressions, position, age, topic, and client, ideally using performance before and after the refresh.

So I would treat this result as a strong observed association and useful decision-support evidence, but not as proof that refreshing a page causes a 57x increase in impressions.


### Finding 2 — “AI-Generated Content Is Penalized” — Debunked

The paper reports that it does not observe a blanket penalty for AI-generated content. However, it also states that almost all of the portfolio was AI-generated across several model families.

My methodology question is about the comparison group. If almost all of the content is AI-generated, what human-authored control group allows the study to isolate the effect of AI use itself?

The age-controlled comparison between AI model families is useful for comparing different AI workflows, but it does not by itself test AI-generated content against comparable human-written content.

I would therefore use narrower claim language: within this mostly AI-authored portfolio, the measured results do not show a consistent performance pattern that can be attributed to AI use alone.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I re-evaluated the same **Random Forest — Full Signal** model using a more honest validation design.

The model, features, target, dataset, and evaluation metric remained the same.  
The important change was the validation split.

| Validation design | Precision@100 | Client-disjoint? |
|---|---:|---|
| Random row CV | **90.0%** | No |
| Balanced client-grouped CV | **74.6%** | Yes |

Under random row cross-validation, the model achieved **90.0% mean Precision@100 (SD 1.9%)**.

However, pages from the same clients appeared in both training and validation. On average, about **29.6 clients** were present on both sides of each random-row split. This allows the model to benefit from client-specific patterns it has already seen.

After switching to **client-disjoint grouped validation**, where complete clients are held out from training, mean Precision@100 decreased to **74.6% (SD 13.6%)**.

This is a drop of **15.4 percentage points**.

I consider the **74.6% result more credible**. The lower score is not evidence that the model became worse; it shows that the random-row split was optimistic.

The grouped result better answers the real question:

> Can the model rank future-decline risk for pages from clients it did not see during training?

Therefore, **74.6% Precision@100 is the primary model performance number I report.**

### Reproducibility note

Before finalizing the results, I made the monthly model-frame row order deterministic by sorting on `client_id` and `content_id` after the DuckDB aggregation.

The underlying data definition, features, target, and model settings did not change. The exact Random Forest results moved slightly, so I report the deterministic rerun values above.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I repeated the leakage check from Week 3 on my **final 21-feature set**.

The prediction point is the end of **March 15, 2026**, so every model feature must be available by that date. The outcome window, **March 16–31**, is used only to create the target.

### What I checked

I explicitly excluded:

- `client_id`
- `content_id`
- `imp_future`
- `avg_daily_imp_future`
- `impression_change_pct`
- `is_declining_proxy`

The client and content IDs are used only for grouping and validation, never as model inputs.

All client-relative percentile features are also calculated from **pre-outcome data only**, before filtering pages based on future coverage.

I then checked the final feature list for:

1. direct overlap with forbidden fields; and
2. feature names containing suspicious terms such as `future`, `outcome`, or `target`.

The audit returned:

- **Forbidden overlap:** `[]`
- **Suspicious future-named features:** `[]`
- **Missing feature values:** `0`

### Conclusion

**Leakage guard passed.**

All 21 final model features are derived only from information available in the first half of the month. No future outcome field, target-derived field, client ID, or content ID is used by the model.

This matters because otherwise the model could appear highly accurate simply by seeing information that would not have been available at the real prediction time.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Too strong

> **The Random Forest predicts which pages will decline and clearly outperforms the baseline.**

### Safer, evidence-based claim

Under the **March 2026 client-disjoint 5-fold validation**, the Random Forest — Full Signal model achieved a measured mean **Precision@100 of 74.6% (SD 13.6%)**.

For a fair comparison, I re-evaluated the original Week-4 rule inside the **same client-disjoint validation design**, using the same target and the same Precision@100 metric. Under this evaluation, the Week-4 rule achieved **35.2% (SD 6.4%)**.

This is an observed difference of **+39.4 percentage points** in favor of the learned model.

The **35.2% value is not the original Week-4 result of 39.0%** reported when the rule ranked the full March queue once. It is the same Week-4 rule re-evaluated under the Week-5 grouped validation design so that the comparison is like-for-like.

However, the rule baseline is not the only relevant comparison.

A stronger **momentum-only Random Forest** achieved **66.2% Precision@100 (SD 12.8%)** under the same client-disjoint validation design. The full-signal model therefore showed a smaller observed improvement of **+8.4 percentage points** over the momentum-only model.

This second comparison is important. It suggests that the additional context, trend-shape, CTR, position, and client-relative features provide useful ranking information beyond recent momentum alone. However, fold-level variability remains substantial, so I do not interpret the +8.4-point difference as proof that the full feature set will always outperform the simpler momentum model.

A separate client-level check showed the same directional pattern using **macro-client Precision@10**:

- Random Forest — Full Signal: **69.2%**
- Random Forest — Momentum Only: **58.8%**
- Week-4 Rule Baseline: **38.8%**

This client-level metric is reported separately because it is **Precision@10 per client**, not the same metric as the fold-level Precision@100 above. It provides additional evidence about performance across clients, but should not be directly combined with the Precision@100 estimates.

### Interpretation

The defensible conclusion is:

> **Within the evaluated March 2026 data, the full-signal Random Forest produced a materially stronger ranking than the Week-4 rule under the same client-disjoint validation design. It also showed a smaller, directional improvement over a stronger momentum-only model. These results support the model as a decision-support tool for prioritizing pages for review, but they do not establish universal predictive superiority or causal effects.**

The evidence remains limited by the use of a **future-impression-decline proxy**, a single primary month, unequal client sizes, and substantial variation across client folds.

I therefore do not claim that the model predicts Google's ranking algorithm, that its important features cause future decline, or that the measured performance will generalize unchanged to new clients and future periods. Additional temporal and prospective validation would be required before making a production-level claim.

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.